In [ ]:
# BAGIAN A: Memuat Data dan Library

# Kita impor semua library yang akan kita pakai, baik untuk demo maupun untuk pipeline penuh.

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline

# Impor SEMUA transformer yg kita butuhkan
from feature_engine.imputation import (
    CategoricalImputer, 
    MeanMedianImputer,
    ArbitraryNumberImputer
)
from feature_engine.encoding import (
    RareLabelEncoder, 
    OneHotEncoder
)

# Muat data mentah SEKALI saja
data = pd.read_csv('data/train.csv')

print("Library dan data siap.")

Library dan data siap.


In [2]:
# BAGIAN B: Menganalisa Kejanggalan Awal

# Sebelum kita proses, kita lihat dulu "kejanggalan" atau masalah di data mentah.

In [ ]:
### Kejanggalan 1: Data Hilang (NaN)

# Ini adalah masalah terbesar. Model machine learning tidak bisa memproses data `NaN`.

In [3]:
# Cek jumlah NaN di setiap kolom
missing_data = data.isnull().sum()

# Tampilkan hanya kolom yang PUNYA data NaN, urutkan dari yg terbanyak
print(f"Total NaN di dataset: {missing_data.sum()}")
print("\nKolom dengan data NaN:")
print(missing_data[missing_data > 0].sort_values(ascending=False))

Total NaN di dataset: 7829

Kolom dengan data NaN:
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


In [ ]:
### Kejanggalan 2: Tipe Data Salah

# Ada kolom yang kelihatannya angka (int64), tapi sebenarnya adalah kategori.

In [4]:
# Tampilkan info ringkas dataset
# Perhatikan MSSubClass, OverallCond, YrSold, MoSold -> Semuanya int64 (angka)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [ ]:
### Kejanggalan 3: Kategori Terlalu Banyak

# Kolom kategorial seperti 'Neighborhood' punya terlalu banyak pilihan unik, ini bisa membuat model pusing.

In [5]:
print(f"Jumlah kategori unik di 'Neighborhood': {data['Neighborhood'].nunique()}")

# Tampilkan 15 teratas
data['Neighborhood'].value_counts().head(15)

Jumlah kategori unik di 'Neighborhood': 25


Neighborhood
NAmes      225
CollgCr    150
OldTown    113
Edwards    100
Somerst     86
Gilbert     79
NridgHt     77
Sawyer      74
NWAmes      73
SawyerW     59
BrkSide     58
Crawfor     51
Mitchel     49
NoRidge     41
Timber      38
Name: count, dtype: int64

In [ ]:
# BAGIAN C: Demo Solusi (Fungsi Feature-Engine)

# Sekarang kita demo *before-after* fungsi `feature-engine` untuk mengatasi masalah tadi.

In [ ]:
### Demo 1: `CategoricalImputer` (Solusi Kejanggalan 1)

In [6]:
print("--- BEFORE ---")
print(f"Total NaNs di 'Alley': {data['Alley'].isnull().sum()}")
# Tampilkan data mentah, kita lihat 'NaN'
display(data[['Alley']].head(10))

# Gunakan transformer
imputer = CategoricalImputer(imputation_method='missing', fill_value='None', variables=['Alley'])
data_transformed = imputer.fit_transform(data)

print("\n--- AFTER ---")
print(f"Total NaNs di 'Alley' setelah diisi: {data_transformed['Alley'].isnull().sum()}")
# Tampilkan data hasil proses, 'NaN' sudah diganti 'None'
display(data_transformed[['Alley']].head(10))

--- BEFORE ---
Total NaNs di 'Alley': 1369


,Alley
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN
6,NaN
7,NaN
8,NaN
9,NaN



--- AFTER ---
Total NaNs di 'Alley' setelah diisi: 0


,Alley
0,None
1,None
2,None
3,None
4,None
5,None
6,None
7,None
8,None
9,None


In [ ]:
### Demo 2: `RareLabelEncoder` (Solusi Kejanggalan 3)

In [7]:
print("--- BEFORE ---")
print(f"Kategori unik: {data['Neighborhood'].nunique()}")
# Tampilkan jumlah per kategori sbg tabel
display(data['Neighborhood'].value_counts().to_frame())

# Gunakan transformer
rare_encoder = RareLabelEncoder(tol=0.05, n_categories=1, variables=['Neighborhood']) # tol=0.05 -> gabung yg < 5%
data_transformed = rare_encoder.fit_transform(data)

print("\n--- AFTER ---")
print(f"Kategori unik baru: {data_transformed['Neighborhood'].nunique()}")
# Tampilkan lagi, kategori yg langka sudah digabung jadi 'Rare'
display(data_transformed['Neighborhood'].value_counts().to_frame())

--- BEFORE ---
Kategori unik: 25


,count
Neighborhood,
NAmes,225
CollgCr,150
OldTown,113
Edwards,100
Somerst,86
Gilbert,79
NridgHt,77
Sawyer,74
NWAmes,73



--- AFTER ---
Kategori unik baru: 10


,count
Neighborhood,
Rare,483
NAmes,225
CollgCr,150
OldTown,113
Edwards,100
Somerst,86
Gilbert,79
NridgHt,77
Sawyer,74


In [ ]:
### Demo 3: `OneHotEncoder` (Solusi Kejanggalan 2)

In [8]:
# Buat data demo kecil
demo_df = pd.DataFrame({'Street': ['Pave', 'Grvl', 'Pave', 'Pave']})

print("--- BEFORE ---")
# Tampilkan tabel mentah (kategori 'Street')
display(demo_df)

# Gunakan transformer
ohe = OneHotEncoder(variables=['Street'], drop_last=False)
data_transformed = ohe.fit_transform(demo_df)

print("\n--- AFTER ---")
# Tampilkan tabel hasil proses (dipecah jadi 2 kolom angka 0/1)
display(data_transformed)

--- BEFORE ---


,Street
0,Pave
1,Grvl
2,Pave
3,Pave



--- AFTER ---


,Street_Pave,Street_Grvl
0,1,0
1,0,1
2,1,0
3,1,0


In [ ]:
# BAGIAN D: Pipeline Penuh (Produksi)

# Sekarang kita gabungkan semua langkah tadi ke dalam 1 Pipeline untuk memproses data `train` dan `test` secara penuh.

In [10]:
# 1. Load Data Mentah (lagi, untuk memastikan bersih)
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

# 2. Pisahkan Fitur (X) dan Target (y)
X_train = train_df.drop(['Id', 'SalePrice'], axis=1)
X_test = test_df.drop('Id', axis=1)
y_train_log = np.log1p(train_df['SalePrice'])

# 3. PERBAIKAN TIPE DATA (Solusi Kejanggalan 2)
# Ubah kolom angka yg sebenarnya kategori jadi 'object' (string)
cols_to_cast = ['MSSubClass', 'OverallCond', 'YrSold', 'MoSold']
for col in cols_to_cast:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)
    
print("Tipe data berhasil diubah (cast).")
print(f"X_train asli: {X_train.shape}, X_test asli: {X_test.shape}")

Tipe data berhasil diubah (cast).
X_train asli: (1460, 79), X_test asli: (1459, 79)


In [11]:
# 4. Definisikan Grup Fitur (Ini bagian "ribet"-nya, tapi cuma 1x)

# Kolom numerik yang 'NA'-nya diisi median
FEATURES_NUM_MEDIAN = ['LotFrontage', 'GarageYrBlt']

# Kolom kategorial yang 'NA'-nya diisi string 'Missing'
FEATURES_CAT_MISSING = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 
    'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
]

# Kolom kategorial yang 'NA'-nya diisi modus (nilai terbanyak)
FEATURES_CAT_MODE = [
    'Electrical', 'KitchenQual', 'Functional', 'SaleType', 'Utilities', 'Exterior1st', 'Exterior2nd', 'MSZoning'
]

# Kumpulkan SEMUA kolom kategorial untuk di-encoding
FEATURES_CAT_ALL = FEATURES_CAT_MISSING + FEATURES_CAT_MODE + cols_to_cast # cols_to_cast dari sel sebelumnya

# Kolom numerik sisanya (kita akan biarkan, tapi 'NA' harus diisi)
FEATURES_NUM_ALL = [col for col in X_train.select_dtypes(include='number') 
                    if col not in FEATURES_NUM_MEDIAN]

print("Grup fitur berhasil didefinisikan.")

Grup fitur berhasil didefinisikan.


In [12]:
# 5. Buat Pipeline (Resep Preprocessing)
house_price_pipeline = Pipeline([

    # --- Tahap 1: Imputasi (Isi data kosong) ---
    ('num_imputer_median', MeanMedianImputer(imputation_method='median', variables=FEATURES_NUM_MEDIAN)),
    ('cat_imputer_missing', CategoricalImputer(imputation_method='missing', fill_value='Missing', variables=FEATURES_CAT_MISSING)),
    ('cat_imputer_mode', CategoricalImputer(imputation_method='frequent', variables=FEATURES_CAT_MODE)),
    ('num_imputer_zero', ArbitraryNumberImputer(arbitrary_number=0, variables=FEATURES_NUM_ALL)),
    
    # --- Tahap 2: Encoding (Ubah jadi angka) ---
    ('rare_label_encoder', RareLabelEncoder(tol=0.01, n_categories=1, variables=FEATURES_CAT_ALL)),
    ('one_hot_encoder', OneHotEncoder(drop_last=True, variables=FEATURES_CAT_ALL)),
])

print("Pipeline preprocessing berhasil dibuat.")

Pipeline preprocessing berhasil dibuat.


In [13]:
# 6. Jalankan Pipeline
print("Fitting pipeline...")
house_price_pipeline.fit(X_train)

print("Transforming train and test data...")
X_train_processed = house_price_pipeline.transform(X_train)
X_test_processed = house_price_pipeline.transform(X_test)

print("Data berhasil diproses.")

Fitting pipeline...
Transforming train and test data...
Data berhasil diproses.


In [14]:
# 7. Simpan Hasil ke `output/`
train_processed_final = X_train_processed.copy()
train_processed_final['SalePrice_Log'] = y_train_log

train_processed_final.to_csv('output/train_processed.csv', index=False)
X_test_processed.to_csv('output/test_processed.csv', index=False)

print("File 'train_processed.csv' dan 'test_processed.csv' berhasil disimpan di folder 'output/'.")

File 'train_processed.csv' dan 'test_processed.csv' berhasil disimpan di folder 'output/'.


In [ ]:
# BAGIAN E: Perbandingan Akhir (Full Dataset)

# Mari kita bandingkan data `X_train` (mentah) dengan `X_train_processed` (hasil pipeline).

In [15]:
print("--- PERBANDINGAN DATA HILANG (NaN) ---")
print(f"Total NaN di X_train (Before): {X_train.isnull().sum().sum()}")
print(f"Total NaN di X_train_processed (After): {X_train_processed.isnull().sum().sum()}")

--- PERBANDINGAN DATA HILANG (NaN) ---
Total NaN di X_train (Before): 7829
Total NaN di X_train_processed (After): 0


In [16]:
print("--- PERBANDINGAN BENTUK DATA (Shape) ---")
print(f"Bentuk X_train (Before): {X_train.shape}")
print(f"Bentuk X_train_processed (After): {X_train_processed.shape}")

--- PERBANDINGAN BENTUK DATA (Shape) ---
Bentuk X_train (Before): (1460, 79)
Bentuk X_train_processed (After): (1460, 179)


In [ ]:
### Perbandingan Tipe Data dan Contoh Tabel

In [17]:
print("--- DATA SEBELUM DIPROSES (Contoh) ---")
# Tampilkan beberapa kolom mentah yg bermasalah
display(X_train[['Alley', 'Neighborhood', 'MSSubClass', 'LotFrontage']].head())

--- DATA SEBELUM DIPROSES (Contoh) ---


,Alley,Neighborhood,MSSubClass,LotFrontage
0,NaN,CollgCr,60,65.0
1,NaN,Veenker,20,80.0
2,NaN,CollgCr,60,68.0
3,NaN,Crawfor,70,60.0
4,NaN,NoRidge,60,84.0


In [18]:
print("\n--- DATA SETELAH DIPROSES (Contoh) ---")
# Tampilkan kolom-kolom yang SUDAH diproses
# Kita cari kolom-kolom yg relevan dari hasil proses
processed_cols_to_show = [
    'LotFrontage', # Hasil imputasi median
    'Alley_Missing', # Hasil imputasi 'Missing' & OHE
    'Alley_Pave', # Hasil OHE
    'Neighborhood_CollgCr', # Hasil OHE
    'Neighborhood_Rare', # Hasil RareLabel & OHE
    'MSSubClass_60' # Hasil Cast & OHE
]

# Ambil hanya kolom yg ada di df hasil proses
existing_cols = [col for col in processed_cols_to_show if col in X_train_processed.columns]
display(X_train_processed[existing_cols].head())


--- DATA SETELAH DIPROSES (Contoh) ---


,LotFrontage,Alley_Missing,MSSubClass_60
0,65.0,1,1
1,80.0,1,0
2,68.0,1,1
3,60.0,1,0
4,84.0,1,1
